# 🧪 Session 9 exercises
### Classification in practice

Session 8 fitted a classifier and scored it. This session is about the decisions around it: what happens when the thing you predict is rare, where the threshold should sit once each mistake has a price, whether the probabilities mean what they say, what changes with three classes, and how a second kind of classifier compares.

Two tables, the same two as the lecture. The **credit table** has 30,000 borrowers and one column saying whether each defaulted. The **index table** is the one from Session 6, with returns in percent.

## How to use this notebook

- Run the **setup cell** below first. It loads both tables, makes the splits and the labels, and imports the scikit-learn pieces.
- Each exercise has a **task**, then a **code cell** for your work. Cells with `...` are blanks to fill in. Replace them with real code.
- Stuck? Open the **💡 Hint**, but only after a genuine attempt. Open the **✅ Solution** to *check* yourself, not to skip the thinking.
- Every cell runs cleanly even with the blanks still in place, so pressing **Run all** never floods you with errors.
- Most exercises stand alone. A few short runs build on each other (B4 to B5, D3 to D4, F3 to F5, G3 to G7, H3 to H4); the task says which earlier exercise it continues from. Section J uses the function from J1, and **section K is five small cases that each start from scratch**. Six of them ask you to draw something: A7, D7, E5, F6, G9 and H8.

**You are not expected to finish all of these.** Do what you can, and come back to the rest when you revise. Short on time? Read the hint, then the solution. A worked solution you genuinely understand is real learning too.

**Units.** The index table is in percent, as in the lecture. The credit table is in New Taiwan dollars, as it comes. Labels have no units at all.

---

## 🧰 Toolkit

New this session. Hover a name for what it does.

<span title="Shuffle the rows and cut them in two. test_size is the share held back, random_state fixes which rows, stratify keeps a label share equal in both halves.">`train_test_split`</span> · <span title="An argument of LogisticRegression. balanced counts each class in inverse proportion to its size while fitting.">`class_weight`</span> · <span title="Sort every value into one of the ranges given. Returns the range each row fell in, or a name if labels= is passed.">`pd.cut`</span> · <span title="Mean squared error of the probabilities, with what happened as 1 or 0. Smaller is better.">`brier_score_loss`</span> · <span title="Wraps a model and learns a small second model mapping its numbers onto probabilities, using folds.">`CalibratedClassifierCV`</span> · <span title="Summarises the whole precision-recall curve in one number. Compare it with the share of ones.">`average_precision_score`</span> · <span title="Prints precision, recall and F1 for every class, plus the macro and weighted averages.">`classification_report`</span> · <span title="average=None gives one score per class, macro averages them equally, weighted averages them by class size.">`f1_score(average=)`</span> · <span title="Classifies a row by a vote among its k nearest rows. Needs a scaler in front of it.">`KNeighborsClassifier`</span> · <span title="The raw score per class, before the exponential and the division that turn scores into probabilities.">`decision_function`</span> · <span title="Flattens a 2 by 2 confusion matrix into tn, fp, fn, tp in reading order.">`.ravel()`</span>

**Formulas**

- cost of a set of decisions = (cost of a miss) x misses + (cost of a false alarm) x false alarms
- the cheapest threshold = cost of a false alarm / (cost of a false alarm + cost of a miss)
- softmax: probability of class *j* = exp(score *j*) divided by the sum of exp over all classes
- a model is calibrated when, among the rows it gave about *p*, about *p* of them happened

---

## ⚙️ Setup · run me first

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             roc_auc_score, average_precision_score, brier_score_loss,
                             f1_score, classification_report)
from sklearn.model_selection import (train_test_split, cross_val_score, TimeSeriesSplit,
                                     GridSearchCV)

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


# ---- the credit table: one row per borrower, no time order ----
credit = pd.read_csv(data_path("credit.csv"))
c_cols = ["limit", "age", "late_now", "months_late", "bill", "paid", "utilisation"]
c_train, c_test = train_test_split(credit, test_size=0.3, random_state=0,
                                   stratify=credit["default"])

# ---- the eleven instruments, for section J. Returns in percent. ----
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
tickers = sorted(prices["ticker"].unique())
rets = prices.pivot(index="date", columns="ticker", values="close").pct_change().dropna() * 100

# ---- the index table from Session 6: one row per trading day, percent ----
table = pd.read_csv(data_path("market_features.csv"), parse_dates=["date"]).set_index("date")
columns = list(table.columns[:19])
ratio = table["vol_next"] / table["vol_20d"]

table["rising"] = (table["vol_next"] > table["vol_20d"]).astype(int)   # Session 8's label
table["jump"]   = (ratio > 1.5).astype(int)                            # a RARE label
table["move"]   = pd.cut(ratio, [-np.inf, 0.85, 1.25, np.inf],
                         labels=["falls", "stays", "rises"])           # three classes

train = table.loc[:"2022-12-31"]
test = table.loc["2023-01-01":]
folds = TimeSeriesSplit(n_splits=5)

print("credit:", credit.shape, "  default share", round(credit["default"].mean(), 4))
print("prices:", prices.shape[0], "rows,", len(tickers), "instruments")
print("index :", table.shape, "  train", len(train), " test", len(test))
print("jump share: train", round(train["jump"].mean(), 3), " test", round(test["jump"].mean(), 3))
print("move counts in the training days:")
print(train["move"].value_counts())

---

## A · Rare labels, and the rule to beat

A label is only as interesting as the baseline it has to beat. These build the baselines.

### A1 · The share that defaulted  ★☆☆☆☆  · revisits S1

Print the number of borrowers who defaulted and the share, the share formatted as a percentage with one decimal.

In [ ]:
n_defaults = ...
share_text = ...

print(n_defaults)
print(share_text)

<details>
<summary>💡 Hint</summary>

The label is a column of ones and zeros, so `.sum()` counts them and `.mean()` is the share. Format it with `f"{value:.1%}"`.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_defaults = credit['default'].sum()
share_text = f"{credit['default'].mean():.1%}"

print(n_defaults)
print(share_text)
```

6,636 of 30,000 borrowers defaulted, which is 22.1%. One borrower in five is not rare in the way fraud is, but it is far enough from half that accuracy stops being useful on its own.

</details>

---

### A2 · The rule to beat  ★★☆☆☆

Without fitting anything, work out the accuracy of predicting that nobody defaults, on the test borrowers. Build the prediction with `np.zeros` and score it with `accuracy_score` rather than reasoning it out.

In [ ]:
nobody = ...
baseline = ...

print(baseline)

<details>
<summary>💡 Hint 1</summary>

`np.zeros(len(c_test), dtype=int)` is a prediction of 0 on every row.

</details>

<details>
<summary>💡 Hint 2</summary>

`accuracy_score(c_test['default'], nobody)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
nobody = np.zeros(len(c_test), dtype=int)
baseline = round(accuracy_score(c_test['default'], nobody), 4)

print(baseline)
```

0.7788. A model that scores 0.79 here has done nothing at all. Computing the baseline rather than assuming it is worth the two lines, because it is the number every later score is read against.

</details>

---

### A3 · Default rate by months late  ★★☆☆☆  · revisits S3

Group the borrowers by `months_late` and report, for each value, how many borrowers there are and what share of them defaulted, rounded to three decimals.

In [ ]:
by_late = ...

print(by_late)

<details>
<summary>💡 Hint</summary>

`credit.groupby('months_late')['default'].agg(['size', 'mean'])`, then `.round(3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_late = credit.groupby('months_late')['default'].agg(['size', 'mean']).round(3)

print(by_late)
```

The default rate climbs from 0.117 for borrowers who were never late to 0.703 for those late in all six months. There is a signal in the table, which is worth confirming before fitting anything to it.

</details>

---

### A4 · A rare label of your own  ★★☆☆☆

On the index table, make a label for a calm month: `ratio` below 0.7, meaning next month's volatility is at least 30 percent lower than this month's. Report how many such days there are and their share. `ratio` comes from the setup cell.

In [ ]:
calm = ...
n_calm = ...
share_calm = ...

print(n_calm)
print(share_calm)

<details>
<summary>💡 Hint</summary>

The same shape as the `jump` label in the setup cell, with `<` instead of `>` and a different number.

</details>

<details>
<summary>✅ Solution</summary>

```python
table['calm'] = (ratio < 0.7).astype(int)
calm = table['calm']
n_calm = int(calm.sum())
share_calm = round(float(calm.mean()), 4)

print(n_calm)
print(share_calm)
```

519 days, or 21.8% of them. Any comparison makes a label, and where you put the cut decides how rare it is. A cut that leaves too few ones cannot be scored at all.

</details>

---

### A5 · The same share, counted by hand  ★★☆☆☆  · revisits S2

Count the jumps among the test days with a `for` loop and an `if`, then check your count against the vectorised `.sum()`. Print both.

In [ ]:
count = 0
for value in test['jump']:
    ...

print(count)
print(test['jump'].sum())

<details>
<summary>💡 Hint</summary>

Inside the loop: `if value == 1:` and then `count += 1`.

</details>

<details>
<summary>✅ Solution</summary>

```python
count = 0
for value in test['jump']:
    if value == 1:
        count += 1

print(count)
print(test['jump'].sum())
```

Both print 36. The loop is the definition and `.sum()` is the fast way to say the same thing. When a vectorised line surprises you, writing the loop once is how you find out which of you is wrong.

</details>

---

### A6 · Shares into a dictionary  ★★☆☆☆  · revisits S2

Build a dictionary with one entry per label, holding the share of test days on which it is 1: the keys `'rising'` and `'jump'`, the values their means. Then print the key with the smaller share.

In [ ]:
shares = {}
for name in ['rising', 'jump']:
    ...

smaller = ...

print(shares)
print(smaller)

<details>
<summary>💡 Hint 1</summary>

Inside the loop: `shares[name] = test[name].mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(dictionary, key=dictionary.get)` gives the KEY with the smallest value, from Session 5.

</details>

<details>
<summary>✅ Solution</summary>

```python
shares = {}
for name in ['rising', 'jump']:
    shares[name] = round(float(test[name].mean()), 3)

smaller = min(shares, key=shares.get)

print(shares)
print(smaller)
```

`rising` is 1 on 0.467 of the test days and `jump` on 0.075, so `jump` is the smaller. Both labels come from the same ratio, and only the cut differs.

</details>

---

### A7 · Draw the default rate  ★☆☆☆☆  · revisits S3

Draw the default rate by `months_late` as a bar chart, with a dashed horizontal line at the rate for everyone. Label both axes.

In [ ]:
by_late = credit.groupby('months_late')['default'].mean()

fig, ax = plt.subplots(figsize=(7, 3))
...
...
ax.set_xlabel('months of the last six that were late')
ax.set_ylabel('share who defaulted')
plt.show()

<details>
<summary>💡 Hint</summary>

`ax.bar(by_late.index, by_late.values)` and `ax.axhline(credit['default'].mean(), linestyle='--', color='grey')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_late = credit.groupby('months_late')['default'].mean()

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(by_late.index, by_late.values)
ax.axhline(credit['default'].mean(), linestyle='--', color='grey')
ax.set_xlabel('months of the last six that were late')
ax.set_ylabel('share who defaulted')
ax.set_title('Default rate by months late', loc='left')
plt.show()
```

Every bar from one month late upwards sits above the dashed line at 0.22, and the climb runs from 0.12 to 0.70. A3 gave you these numbers; the picture is what makes the shape of them obvious at a glance.

</details>

---

---

## B · Splitting rows that are not a time series

The credit rows are borrowers, not days, so they can be shuffled. B4 and B5 run together.

### B1 · A stratified split  ★★☆☆☆

Split the credit table into 70 percent training rows and 30 percent test rows, with `random_state=0`, keeping the share of defaults equal in both halves. Print the two sizes and the two shares.

In [ ]:
...
sizes = ...
shares = ...

print(sizes)
print(shares)

<details>
<summary>💡 Hint</summary>

`train_test_split(credit, test_size=0.3, random_state=0, stratify=credit['default'])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
a, b = train_test_split(credit, test_size=0.3, random_state=0,
                        stratify=credit['default'])
sizes = (len(a), len(b))
shares = (round(float(a['default'].mean()), 4), round(float(b['default'].mean()), 4))

print(sizes)
print(shares)
```

21,000 and 9,000 rows, both with a default share of 0.2212. This is the split the setup cell already made, so `a` and `b` hold the same borrowers as `c_train` and `c_test`.

</details>

---

### B2 · What stratify is for  ★★★☆☆

Take a deliberately small test set, 2 percent of the borrowers, five times with `random_state` 0 to 4, once with `stratify` and once without. Collect the default share of the test set each time and print both lists.

In [ ]:
with_strat = []
without = []
for seed in range(5):
    _, s = train_test_split(credit, test_size=0.02, random_state=seed,
                            stratify=credit['default'])
    _, u = train_test_split(credit, test_size=0.02, random_state=seed)
    ...
    ...

print(with_strat)
print(without)

<details>
<summary>💡 Hint</summary>

`with_strat.append(round(s['default'].mean(), 4))`, and the same for `u` into `without`.

</details>

<details>
<summary>✅ Solution</summary>

```python
with_strat = []
without = []
for seed in range(5):
    _, s = train_test_split(credit, test_size=0.02, random_state=seed,
                            stratify=credit['default'])
    _, u = train_test_split(credit, test_size=0.02, random_state=seed)
    with_strat.append(round(s['default'].mean(), 4))
    without.append(round(u['default'].mean(), 4))

print(with_strat)
print(without)
```

With `stratify` every share is 0.2217. Without it they run from 0.2133 to 0.255, a spread of 0.0417. On 9,000 test rows that drift is small, but on a small sample of a rare class it is large enough to move every score you report.

</details>

---

### B3 · The split that would be wrong here  ★★★☆☆  · revisits S5

The index table is a time series, so it is split by date instead. Make that split, then print the two sizes and, beside them, the last training date and the first test date.

In [ ]:
early = ...
late = ...
sizes = ...
ends = ...

print(sizes)
print(ends)

<details>
<summary>💡 Hint</summary>

`table.loc[:'2022-12-31']` and `table.loc['2023-01-01':]`, as in Session 5.

</details>

<details>
<summary>✅ Solution</summary>

```python
early = table.loc[:'2022-12-31']
late = table.loc['2023-01-01':]
sizes = (len(early), len(late))
ends = (early.index.max().date(), late.index.min().date())

print(sizes)
print(ends)
```

1,894 training days and 482 test days, with every test day after every training day. A shuffled split here would let the model fit on 2024 and be scored on 2020.

</details>

---

### B4 · Fit the credit classifier  ★★☆☆☆

Fit a scaled logistic regression on the credit training rows, using all seven columns, and store the probability of default for the test rows in `p`. Print the accuracy of `.predict()` and the baseline from A2.

In [ ]:
model = ...
...
p = ...
accuracy = ...

print(accuracy)
print(round(1 - c_test['default'].mean(), 4))

<details>
<summary>💡 Hint 1</summary>

`Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])`.

</details>

<details>
<summary>💡 Hint 2</summary>

`model.predict_proba(c_test[c_cols])[:, 1]` is the probability of a 1.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]
accuracy = round(accuracy_score(c_test['default'], model.predict(c_test[c_cols])), 4)

print(accuracy)
print(round(1 - c_test['default'].mean(), 4))
```

0.8150 against 0.7788, so the model is 3.6 percentage points better than predicting that nobody defaults. Keep `model` and `p`: B5 continues from here.

</details>

---

### B5 · The four counts  ★★☆☆☆  · revisits S4

Continuing from B4. Print the confusion matrix, then unpack it with `.ravel()` into the four counts and print them with labels.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

cm = ...
counts = ...

print(cm)
print(counts)

<details>
<summary>💡 Hint</summary>

`confusion_matrix(c_test['default'], predicted)`, then `tn, fp, fn, tp = cm.ravel()`, and collect the four into a dictionary with `int()` around each so they print as plain numbers.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

cm = confusion_matrix(c_test['default'], predicted)
tn, fp, fn, tp = cm.ravel()
counts = {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

print(cm)
print(counts)
```

tn 6,715, fp 294, fn 1,371, tp 620. The model called 914 defaults and was right about 620 of them, while 1,371 borrowers defaulted without being called. `.ravel()` gives the four counts in reading order, which is the order every formula below uses.

</details>

---

---

## C · Accuracy, precision and recall

Four counts, and the numbers built from them. Several of these rebuild by hand what scikit-learn does in one call.

### C1 · Precision and recall  ★☆☆☆☆

Fit the credit model, predict the test rows, and print the precision and the recall, each to three decimals.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

precision = ...
recall = ...

print(precision)
print(recall)

<details>
<summary>💡 Hint</summary>

`precision_score(c_test['default'], predicted)` and `recall_score(...)`, true labels first.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

precision = round(precision_score(c_test['default'], predicted), 3)
recall = round(recall_score(c_test['default'], predicted), 3)

print(precision)
print(recall)
```

Precision 0.678, recall 0.311. Of the borrowers it called, two thirds defaulted, but it found under a third of the defaults. On a rare class those two numbers are the report, not the accuracy.

</details>

---

### C2 · The same two, from masks  ★★★☆☆  · revisits S4

Compute the same precision and recall from boolean masks, with no metric function: build `said` and `was` as columns of `True` and `False`, count the three combinations you need, and divide.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

said = predicted == 1
was = c_test['default'].values == 1

tp = ...
fp = ...
fn = ...
precision = ...
recall = ...

print(precision)
print(recall)

<details>
<summary>💡 Hint 1</summary>

`(said & was).sum()` counts the rows where both are True.

</details>

<details>
<summary>💡 Hint 2</summary>

A false alarm is `said & ~was`, and a miss is `~said & was`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

said = predicted == 1
was = c_test['default'].values == 1

tp = (said & was).sum()
fp = (said & ~was).sum()
fn = (~said & was).sum()
precision = round(tp / (tp + fp), 3)
recall = round(tp / (tp + fn), 3)

print(precision)
print(recall)
```

The same 0.678 and 0.311 as C1. Session 4 counted these by hand before any model existed. The metric functions are a shorthand for exactly this, and knowing that is what lets you debug one that surprises you.

</details>

---

### C3 · A function for the four counts  ★★★★☆  · revisits S2

Write `counts(y_true, y_pred)` returning a dictionary with the keys `'tp'`, `'fp'`, `'fn'` and `'tn'`. Give it a docstring, and test it on the credit predictions.

In [ ]:
def counts(y_true, y_pred):
    """..."""
    ...


model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

print(counts(c_test['default'], predicted))

<details>
<summary>💡 Hint 1</summary>

Inside the function: `tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()`.

</details>

<details>
<summary>💡 Hint 2</summary>

Return `{'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)}`. `int()` keeps the printed dictionary readable.

</details>

<details>
<summary>✅ Solution</summary>

```python
def counts(y_true, y_pred):
    """The four counts of a 0/1 prediction, as a dictionary."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)}


model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

print(counts(c_test['default'], predicted))
```

`{'tp': 620, 'fp': 294, 'fn': 1371, 'tn': 6715}`. A function that returns a dictionary is the tidy way to hand several numbers back at once, and it makes the exercises below one line each.

</details>

---

### C4 · Recall at several thresholds  ★★☆☆☆  · revisits S2

Loop over the thresholds 0.1, 0.2, 0.3, 0.4 and 0.5 and print each one with the recall it gives, rounded to three decimals.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    called = ...
    ...

<details>
<summary>💡 Hint</summary>

`called = (p >= threshold).astype(int)`, then `print(threshold, round(recall_score(c_test['default'], called), 3))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    called = (p >= threshold).astype(int)
    print(threshold, round(recall_score(c_test['default'], called), 3))
```

Recall falls from 0.953 at 0.1 to 0.311 at 0.5. The model never changed; only the number its probabilities are compared against did.

</details>

---

### C5 · A table that lines up  ★★★☆☆  · revisits S2

Print one line per threshold with the threshold, the precision and the recall, using field widths so the columns line up under the header.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

print(f"{'thr':>5}{'prec':>8}{'recall':>8}")
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    called = (p >= threshold).astype(int)
    prec = precision_score(c_test['default'], called, zero_division=0)
    rec = recall_score(c_test['default'], called)
    ...

<details>
<summary>💡 Hint</summary>

`print(f"{threshold:>5}{prec:>8.3f}{rec:>8.3f}")`. The number after the colon is the width, and `>` right-aligns.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

print(f"{'thr':>5}{'prec':>8}{'recall':>8}")
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    called = (p >= threshold).astype(int)
    prec = precision_score(c_test['default'], called, zero_division=0)
    rec = recall_score(c_test['default'], called)
    print(f"{threshold:>5}{prec:>8.3f}{rec:>8.3f}")
```

Precision climbs as recall falls, and a column that lines up is the difference between seeing that and not. Field widths came up in Session 2 and are worth the extra characters every time you print inside a loop.

</details>

---

### C6 · Walking down to a recall of one half  ★★★★☆  · revisits S2

Start at a threshold of 0.5 and step down by 0.05 at a time. Stop at the first threshold whose recall reaches 0.5, and print it with its recall. Use `while` and `break`.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

threshold = 0.5
rec = 0.0
while threshold > 0.05:
    rec = recall_score(c_test['default'], (p >= threshold).astype(int))
    if ...:
        break
    threshold = round(threshold - 0.05, 2)

print(threshold, round(rec, 3))

<details>
<summary>💡 Hint</summary>

The condition is `rec >= 0.5`. The `while` is bounded, so the loop always stops even if the condition is never met.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

threshold = 0.5
rec = 0.0
while threshold > 0.05:
    rec = recall_score(c_test['default'], (p >= threshold).astype(int))
    if rec >= 0.5:
        break
    threshold = round(threshold - 0.05, 2)

print(threshold, round(rec, 3))
```

It stops at 0.25, where the recall is 0.524. A `while` with a `break` is the right shape when you are looking for the first value that satisfies something, and the bound on the `while` is what stops it running forever when nothing does.

</details>

---

---

## D · Thresholds and costs

A missed default and a false alarm are not worth the same, so the threshold that makes the fewest mistakes is not the cheapest one. D3 and D4 run together. Throughout: a miss costs 5, a false alarm costs 1.

### D1 · The cost of a threshold  ★★☆☆☆

Write the cost of the test decisions at a threshold of 0.5, counting 5 for every missed default and 1 for every false alarm.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

called = (p >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(c_test['default'], called).ravel()
cost = ...

print(fn, fp, cost)

<details>
<summary>💡 Hint</summary>

A miss is `fn` and a false alarm is `fp`, so the cost is `5 * fn + fp`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

called = (p >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(c_test['default'], called).ravel()
cost = 5 * fn + fp

print(fn, fp, cost)
```

1,371 misses and 294 false alarms, costing 7,149. Almost all of the cost is misses, which is the clue that one half is the wrong threshold here.

</details>

---

### D2 · Sweeping the threshold  ★★★☆☆  · revisits S3

Sweep the threshold from 0.05 to 0.85 in steps of 0.01 with `np.arange`, collect the cost at each one in a list, and print the cheapest threshold and its cost. Use `np.array` and `.argmin()`.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

grid = np.arange(0.05, 0.85, 0.01)
costs = []
for threshold in grid:
    tn, fp, fn, tp = confusion_matrix(c_test['default'], (p >= threshold).astype(int)).ravel()
    ...

costs = np.array(costs)
best = ...
cheapest = ...

print(best)
print(cheapest)

<details>
<summary>💡 Hint 1</summary>

Inside the loop: `costs.append(5 * fn + fp)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`.argmin()` gives the POSITION of the smallest cost, so `best = grid[costs.argmin()]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

grid = np.arange(0.05, 0.85, 0.01)
costs = []
for threshold in grid:
    tn, fp, fn, tp = confusion_matrix(c_test['default'], (p >= threshold).astype(int)).ravel()
    costs.append(5 * fn + fp)

costs = np.array(costs)
best = round(float(grid[costs.argmin()]), 2)
cheapest = int(costs.min())

print(best)
print(cheapest)
```

0.18, costing 5,373 against 7,149 at one half. Moving one number saved 25 percent of the cost, with no change to the model at all.

</details>

---

### D3 · The threshold from the formula  ★★☆☆☆

The cheapest threshold can be written down without a sweep: the cost of a false alarm divided by the two costs added together. Compute it for a miss of 5 and a false alarm of 1, and print the cost it gives.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]
cost_miss = 5
cost_false = 1

threshold = ...
cost_here = ...

print(threshold)
print(cost_here)

<details>
<summary>💡 Hint</summary>

`cost_false / (cost_false + cost_miss)`, which is one sixth.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]
cost_miss = 5
cost_false = 1

threshold = cost_false / (cost_false + cost_miss)
tn, fp, fn, tp = confusion_matrix(c_test['default'], (p >= threshold).astype(int)).ravel()
cost_here = 5 * fn + fp

print(round(threshold, 3))
print(cost_here)
```

0.167, costing 5,386 against the 5,373 the sweep found. The formula lands within half a percent of the best available, and it needs no test rows to find it. D4 continues from here.

</details>

---

### D4 · Choosing it on the training rows  ★★★★☆

Continuing from D3. The test rows are not available when the threshold is chosen, so sweep on the **training** rows instead, then report what that threshold costs on the test rows.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p_train = model.predict_proba(c_train[c_cols])[:, 1]
p = model.predict_proba(c_test[c_cols])[:, 1]
grid = np.arange(0.05, 0.85, 0.01)

chosen = ...
cost_on_test = ...

print(chosen)
print(cost_on_test)

<details>
<summary>💡 Hint</summary>

`p_train = model.predict_proba(c_train[c_cols])[:, 1]`. Everything else is the sweep from D2 with the training label.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p_train = model.predict_proba(c_train[c_cols])[:, 1]
p = model.predict_proba(c_test[c_cols])[:, 1]
grid = np.arange(0.05, 0.85, 0.01)

costs = []
for threshold in grid:
    tn, fp, fn, tp = confusion_matrix(c_train['default'], (p_train >= threshold).astype(int)).ravel()
    costs.append(5 * fn + fp)

chosen = round(float(grid[np.array(costs).argmin()]), 2)
tn, fp, fn, tp = confusion_matrix(c_test['default'], (p >= chosen).astype(int)).ravel()
cost_on_test = 5 * fn + fp

print(chosen)
print(cost_on_test)
```

The training rows choose 0.16, which costs 5,375 on the test rows against the 5,373 the test rows would have chosen for themselves. The threshold is a setting like `C`: chosen where the model may look, reported where it may not.

</details>

---

### D5 · A cost function with a default  ★★★★☆  · revisits S2

Write `total_cost(y_true, p, threshold, cost_miss=5, cost_false=1)` returning the cost as a plain integer. Call it twice: once with the defaults, once with a miss costing 20.

In [ ]:
def total_cost(y_true, p, threshold, cost_miss=5, cost_false=1):
    """..."""
    ...


model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

print(total_cost(c_test['default'], p, 0.5))
print(total_cost(c_test['default'], p, 0.5, cost_miss=20))

<details>
<summary>💡 Hint 1</summary>

Inside: build `called`, unpack the four counts with `.ravel()`, and return `int(cost_miss * fn + cost_false * fp)`.

</details>

<details>
<summary>💡 Hint 2</summary>

A parameter with an `=` in the `def` line is a default: the caller may leave it out.

</details>

<details>
<summary>✅ Solution</summary>

```python
def total_cost(y_true, p, threshold, cost_miss=5, cost_false=1):
    """What a set of 0/1 decisions at this threshold costs."""
    called = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, called).ravel()
    return int(cost_miss * fn + cost_false * fp)


model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

print(total_cost(c_test['default'], p, 0.5))
print(total_cost(c_test['default'], p, 0.5, cost_miss=20))
```

7,149 with the defaults and 27,714 when a miss costs 20. Defaults let the common case stay short while the unusual one is still reachable, which is exactly how scikit-learn's own arguments work.

</details>

---

### D6 · Which costs make one half right  ★★☆☆☆  · revisits S1

The formula says the cheapest threshold is the cost of a false alarm over the two costs added together. Work out on paper what the two costs must be for that to give 0.5, then check your answer in code with any pair of numbers that fits.

In [ ]:
cost_miss = ...
cost_false = ...
threshold = ...

print(threshold)

<details>
<summary>💡 Hint</summary>

For the fraction to be one half, the numerator has to be half the denominator, so the two costs must be equal.

</details>

<details>
<summary>✅ Solution</summary>

```python
cost_miss = 1
cost_false = 1
threshold = cost_false / (cost_false + cost_miss)

print(threshold)
```

The two costs have to be equal, and any equal pair works. The default threshold of one half is not a neutral choice: it is the claim that a missed default and a wrongly refused customer cost the bank the same, which almost nobody believes.

</details>

---

### D7 · Draw the cost curve  ★★☆☆☆  · revisits S3

Draw the cost of the test decisions against the threshold, with a vertical line at the cheapest threshold. The sweep is the one from D2.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

grid = np.arange(0.05, 0.85, 0.01)
costs = []
for threshold in grid:
    tn, fp, fn, tp = confusion_matrix(c_test['default'], (p >= threshold).astype(int)).ravel()
    costs.append(5 * fn + fp)

fig, ax = plt.subplots(figsize=(7, 3))
...
...
ax.set_xlabel('threshold')
ax.set_ylabel('cost')
plt.show()

<details>
<summary>💡 Hint</summary>

`ax.plot(grid, costs)`, then `ax.axvline(grid[np.array(costs).argmin()], linestyle='--', color='grey')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

grid = np.arange(0.05, 0.85, 0.01)
costs = []
for threshold in grid:
    tn, fp, fn, tp = confusion_matrix(c_test['default'], (p >= threshold).astype(int)).ravel()
    costs.append(5 * fn + fp)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(grid, costs)
ax.axvline(grid[np.array(costs).argmin()], linestyle='--', color='grey')
ax.set_xlabel('threshold')
ax.set_ylabel('cost')
ax.set_title('Cost of the 9,000 test decisions', loc='left')
plt.show()
```

The curve falls steeply, turns at 0.18 and climbs slowly after it. The shape matters as much as the minimum: anywhere between about 0.15 and 0.3 costs nearly the same, so the exact threshold does not need defending to two decimals.

</details>

---

---

## E · Weighting the rare class, and the PR curve

A second way to move the cut, and the curve that reads a rare class properly.

### E1 · class_weight balanced  ★★☆☆☆

Fit the same pipeline with `class_weight='balanced'` on the classifier, then add it to the list in the loop so both models are reported.

In [ ]:
plain = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
plain.fit(c_train[c_cols], c_train['default'])

weighted = ...
...

for name, m in [('plain', plain)]:
    pred = m.predict(c_test[c_cols])
    print(name, round(accuracy_score(c_test['default'], pred), 4),
          round(recall_score(c_test['default'], pred), 3))

<details>
<summary>💡 Hint</summary>

`Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(class_weight='balanced'))])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
plain = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
plain.fit(c_train[c_cols], c_train['default'])

weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])
weighted.fit(c_train[c_cols], c_train['default'])

for name, m in [('plain', plain), ('weighted', weighted)]:
    pred = m.predict(c_test[c_cols])
    print(name, round(accuracy_score(c_test['default'], pred), 4),
          round(recall_score(c_test['default'], pred), 3))
```

Accuracy falls from 0.8150 to 0.7779, which is below the 0.7788 of predicting that nobody defaults, while recall rises from 0.311 to 0.550. The model got worse by one number and more useful by another.

</details>

---

### E2 · What the weights did not change  ★★☆☆☆

Print the AUC of both models. Before running it, decide whether you expect them to differ.

In [ ]:
plain = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
plain.fit(c_train[c_cols], c_train['default'])
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])
weighted.fit(c_train[c_cols], c_train['default'])

auc_plain = ...
auc_weighted = ...

print(auc_plain)
print(auc_weighted)

<details>
<summary>💡 Hint</summary>

`roc_auc_score(c_test['default'], plain.predict_proba(c_test[c_cols])[:, 1])`, and the same for `weighted`.

</details>

<details>
<summary>✅ Solution</summary>

```python
plain = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
plain.fit(c_train[c_cols], c_train['default'])
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])
weighted.fit(c_train[c_cols], c_train['default'])

auc_plain = round(roc_auc_score(c_test['default'],
                                plain.predict_proba(c_test[c_cols])[:, 1]), 4)
auc_weighted = round(roc_auc_score(c_test['default'],
                                   weighted.predict_proba(c_test[c_cols])[:, 1]), 4)

print(auc_plain)
print(auc_weighted)
```

0.7476 and 0.7481, which are the same to three decimals. AUC reads the order of the borrowers, and the weights did not reorder anyone. They moved where the cut falls, which is something a threshold can do too, and more precisely.

</details>

---

### E3 · Average precision against the base rate  ★★☆☆☆  · revisits S4

Print the average precision of the plain model's probabilities and, beside it, the share of test borrowers who defaulted. The second is what guessing at random would score.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

average_precision = ...

print(average_precision)
print(round(c_test['default'].mean(), 4))

<details>
<summary>💡 Hint</summary>

`average_precision_score(c_test['default'], p)`, with the probabilities and not the predictions.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

average_precision = round(average_precision_score(c_test['default'], p), 4)

print(average_precision)
print(round(c_test['default'].mean(), 4))
```

0.5199 against 0.2212. Where an AUC of 0.5 is the no-information line, the no-information line for average precision is the share of ones, which moves with how rare the class is. Always report the two together.

</details>

---

### E4 · Precision at a recall of 0.6  ★★★☆☆

`precision_recall_curve(y, p)` returns three arrays: the precision, the recall, and the thresholds. Use it to find the precision at the point where recall is closest to 0.6. Import it yourself.

In [ ]:
from sklearn.metrics import precision_recall_curve

model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

prec, rec, thresholds = precision_recall_curve(c_test['default'], p)
spot = ...
recall_here = ...
precision_here = ...

print(recall_here)
print(precision_here)

<details>
<summary>💡 Hint 1</summary>

`np.abs(rec - 0.6)` is how far each recall is from 0.6.

</details>

<details>
<summary>💡 Hint 2</summary>

`.argmin()` gives the position of the smallest distance, so `spot = np.abs(rec - 0.6).argmin()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
from sklearn.metrics import precision_recall_curve

model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

prec, rec, thresholds = precision_recall_curve(c_test['default'], p)
spot = np.abs(rec - 0.6).argmin()
recall_here = round(float(rec[spot]), 3)
precision_here = round(float(prec[spot]), 3)

print(recall_here)
print(precision_here)
```

At a recall of about 0.6 the precision is 0.461: catching three defaults in five means being wrong about rather more than half the borrowers you call. That trade is the decision, and the curve is how you see all of it at once.

</details>

---

### E5 · Draw the curve  ★★★☆☆  · revisits S3

Plot precision against recall from E4, with axis labels and a horizontal line at the share of ones. Three steps, as always: make the axes, draw, then say what the reader is looking at.

In [ ]:
from sklearn.metrics import precision_recall_curve

model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]
prec, rec, thresholds = precision_recall_curve(c_test['default'], p)

fig, ax = plt.subplots(figsize=(6, 3.4))
...
...
ax.set_xlabel('recall')
ax.set_ylabel('precision')
ax.set_ylim(0, 1)
plt.show()

<details>
<summary>💡 Hint</summary>

`ax.plot(rec, prec)` and `ax.axhline(c_test['default'].mean(), linestyle='--', color='grey')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
from sklearn.metrics import precision_recall_curve

model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]
prec, rec, thresholds = precision_recall_curve(c_test['default'], p)

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(rec, prec)
ax.axhline(c_test['default'].mean(), linestyle='--', color='grey')
ax.set_xlabel('recall')
ax.set_ylabel('precision')
ax.set_ylim(0, 1)
ax.set_title('Precision against recall', loc='left')
plt.show()
```

The curve starts high and falls away as recall rises, ending at the dashed line, which is where calling every borrower a default would put you. The area under it is the average precision from E3.

</details>

---

---

## F · Calibration

Whether a probability of 0.3 means 0.3. F3 to F5 run together.

### F1 · Bucket the probabilities  ★★☆☆☆  · revisits S3

Put the test borrowers into four buckets by their predicted probability, with the edges 0, 0.2, 0.4, 0.6 and 1.0, and report how many borrowers are in each and what share of them defaulted.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
checked = c_test.copy()
checked['p'] = model.predict_proba(c_test[c_cols])[:, 1]

checked['bucket'] = ...
by_bucket = ...

print(by_bucket)

<details>
<summary>💡 Hint 1</summary>

`pd.cut(checked['p'], [0, 0.2, 0.4, 0.6, 1.0])`.

</details>

<details>
<summary>💡 Hint 2</summary>

`checked.groupby('bucket', observed=True)['default'].agg(['size', 'mean']).round(3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
checked = c_test.copy()
checked['p'] = model.predict_proba(c_test[c_cols])[:, 1]

checked['bucket'] = pd.cut(checked['p'], [0, 0.2, 0.4, 0.6, 1.0])
by_bucket = checked.groupby('bucket', observed=True)['default'].agg(['size', 'mean']).round(3)

print(by_bucket)
```

Each bucket's share falls inside its own range: 0.128, 0.309, 0.561, 0.714. These numbers can be read as probabilities, which is what calibrated means.

</details>

---

### F2 · The average against what happened  ★☆☆☆☆  · revisits S3

Print the average probability the model gave the test borrowers and the share of them that actually defaulted.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

said = ...
happened = ...

print(said)
print(happened)

<details>
<summary>💡 Hint</summary>

`p.mean()` and `c_test['default'].mean()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

said = round(p.mean(), 4)
happened = round(c_test['default'].mean(), 4)

print(said)
print(happened)
```

0.2187 against 0.2212. This is the cheapest calibration check there is, and it is the first thing to run when a probability is about to be multiplied by an amount of money.

</details>

---

### F3 · The Brier score, both models  ★★☆☆☆

Print the Brier score of the plain model and of the weighted one. Keep both fitted models: F4 and F5 continue from here.

In [ ]:
plain = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
plain.fit(c_train[c_cols], c_train['default'])
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])
weighted.fit(c_train[c_cols], c_train['default'])

brier_plain = ...
brier_weighted = ...

print(brier_plain)
print(brier_weighted)

<details>
<summary>💡 Hint</summary>

`brier_score_loss(c_test['default'], plain.predict_proba(c_test[c_cols])[:, 1])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
plain = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
plain.fit(c_train[c_cols], c_train['default'])
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])
weighted.fit(c_train[c_cols], c_train['default'])

brier_plain = round(brier_score_loss(c_test['default'],
                                     plain.predict_proba(c_test[c_cols])[:, 1]), 4)
brier_weighted = round(brier_score_loss(c_test['default'],
                                        weighted.predict_proba(c_test[c_cols])[:, 1]), 4)

print(brier_plain)
print(brier_weighted)
```

0.1395 for the plain model and 0.1879 for the weighted one. The AUC could not tell them apart, and this can: the weighted model says 0.44 on average where 0.22 happens.

</details>

---

### F4 · Put the level back  ★★★★☆

Continuing from F3. Wrap the weighted pipeline in `CalibratedClassifierCV` with `method='sigmoid'` and `cv=5`, fit it on the training rows, and print the average probability and the Brier score.

In [ ]:
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])

fixed = ...
...
p_fixed = ...

print(p_fixed)

<details>
<summary>💡 Hint</summary>

`CalibratedClassifierCV(weighted, method='sigmoid', cv=5)`. Pass the unfitted pipeline: it refits inside on the folds.

</details>

<details>
<summary>✅ Solution</summary>

```python
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])

fixed = CalibratedClassifierCV(weighted, method='sigmoid', cv=5)
fixed.fit(c_train[c_cols], c_train['default'])
p_fixed = fixed.predict_proba(c_test[c_cols])[:, 1]

print(round(p_fixed.mean(), 4))
print(round(brier_score_loss(c_test['default'], p_fixed), 4))
```

The average comes back to 0.2186 and the Brier score to 0.1395, which is the plain model's. The ranking was never the problem, so calibrating did not need to fix it.

</details>

---

### F5 · What calibration is worth  ★★★★★

Continuing from F4. Apply the cost threshold of one sixth to all three sets of probabilities, and print the cost of each.

In [ ]:
plain = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
plain.fit(c_train[c_cols], c_train['default'])
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])
weighted.fit(c_train[c_cols], c_train['default'])
fixed = CalibratedClassifierCV(Pipeline([('scale', StandardScaler()),
    ('logit', LogisticRegression(class_weight='balanced'))]), method='sigmoid', cv=5)
fixed.fit(c_train[c_cols], c_train['default'])

for name, m in [('plain', plain), ('weighted', weighted), ('fixed', fixed)]:
    prob = m.predict_proba(c_test[c_cols])[:, 1]
    tn, fp, fn, tp = confusion_matrix(c_test['default'], (prob >= 1 / 6).astype(int)).ravel()
    ...

<details>
<summary>💡 Hint</summary>

`print(name, 5 * fn + fp)` inside the loop.

</details>

<details>
<summary>✅ Solution</summary>

```python
plain = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
plain.fit(c_train[c_cols], c_train['default'])
weighted = Pipeline([('scale', StandardScaler()),
                     ('logit', LogisticRegression(class_weight='balanced'))])
weighted.fit(c_train[c_cols], c_train['default'])
fixed = CalibratedClassifierCV(Pipeline([('scale', StandardScaler()),
    ('logit', LogisticRegression(class_weight='balanced'))]), method='sigmoid', cv=5)
fixed.fit(c_train[c_cols], c_train['default'])

for name, m in [('plain', plain), ('weighted', weighted), ('fixed', fixed)]:
    prob = m.predict_proba(c_test[c_cols])[:, 1]
    tn, fp, fn, tp = confusion_matrix(c_test['default'], (prob >= 1 / 6).astype(int)).ravel()
    print(name, 5 * fn + fp)
```

5,386 for the plain model, 6,940 for the weighted one and 5,385 once it is calibrated. The threshold from the cost formula is a statement about probabilities, so it only works on probabilities that mean what they say.

</details>

---

### F6 · Said against happened  ★★★☆☆  · revisits S3

Draw the calibration check: the average probability in each bucket on the horizontal axis, the share that defaulted on the vertical, and a dashed diagonal from (0, 0) to (1, 1) to compare against.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
checked = c_test.copy()
checked['p'] = model.predict_proba(c_test[c_cols])[:, 1]
checked['bucket'] = pd.cut(checked['p'], [0, 0.2, 0.4, 0.6, 1.0])
said = checked.groupby('bucket', observed=True)['p'].mean()
happened = checked.groupby('bucket', observed=True)['default'].mean()

fig, ax = plt.subplots(figsize=(4.2, 4))
...
...
ax.set_xlabel('the model said')
ax.set_ylabel('what happened')
plt.show()

<details>
<summary>💡 Hint</summary>

`ax.plot([0, 1], [0, 1], linestyle='--', color='grey')` for the diagonal, then `ax.plot(said, happened, 'o-')` for the buckets.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
checked = c_test.copy()
checked['p'] = model.predict_proba(c_test[c_cols])[:, 1]
checked['bucket'] = pd.cut(checked['p'], [0, 0.2, 0.4, 0.6, 1.0])
said = checked.groupby('bucket', observed=True)['p'].mean()
happened = checked.groupby('bucket', observed=True)['default'].mean()

fig, ax = plt.subplots(figsize=(4.2, 4))
ax.plot([0, 1], [0, 1], linestyle='--', color='grey')
ax.plot(said, happened, 'o-')
ax.set_xlabel('the model said')
ax.set_ylabel('what happened')
ax.set_title('Four buckets of test borrowers', loc='left')
plt.show()
```

The four points sit close to the diagonal. Run the same plot on the weighted model from F3 and every point drops well below it, which is what a model that overstates its probabilities looks like.

</details>

---

---

## G · Three classes

Back to the index table, with `move`: `falls`, `stays` or `rises`. G3 to G7 run together on the same fitted model.

### G1 · A three-way label of your own  ★★☆☆☆

Make a second three-way label with tighter cuts, 0.9 and 1.1, called `move2`, and print how many training days fall in each class. `ratio` is in the setup cell.

In [ ]:
table['move2'] = ...

print(table.loc[:'2022-12-31', 'move2'].value_counts())

<details>
<summary>💡 Hint</summary>

`pd.cut(ratio, [-np.inf, 0.9, 1.1, np.inf], labels=['falls', 'stays', 'rises'])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
table['move2'] = pd.cut(ratio, [-np.inf, 0.9, 1.1, np.inf],
                        labels=['falls', 'stays', 'rises'])

print(table.loc[:'2022-12-31', 'move2'].value_counts())
```

The middle class is much smaller than with the lecture's cuts of 0.85 and 1.25, because the band is narrower. Where the cuts go decides how big each class is, and a class that is too small cannot be learned or scored.

</details>

---

### G2 · The rule to beat, with three classes  ★★☆☆☆  · revisits S3

Find the most common `move` among the training days, then report the share of TEST days that carry that same label. That is the baseline.

In [ ]:
most_common = ...
baseline = ...

print(most_common)
print(baseline)

<details>
<summary>💡 Hint 1</summary>

`train['move'].value_counts().idxmax()` gives the most common label.

</details>

<details>
<summary>💡 Hint 2</summary>

`(test['move'] == most_common).mean()` is the share of test days it gets right.

</details>

<details>
<summary>✅ Solution</summary>

```python
most_common = train['move'].value_counts().idxmax()
baseline = round((test['move'] == most_common).mean(), 4)

print(most_common)
print(baseline)
```

`falls` is the most common training label, and it is right on 0.3506 of the test days. With three classes the baseline is lower than with two, because guessing is harder, so a model has more room to look good without being good.

</details>

---

### G3 · Fit three classes  ★★☆☆☆

Fit a scaled logistic regression on `vol_20d` and `ret_20d` predicting `move`, with `max_iter=1000`, and print its accuracy on the test days. Keep the model: G4 to G7 continue from here.

In [ ]:
three = ...
...
accuracy = ...

print(accuracy)

<details>
<summary>💡 Hint</summary>

The pipeline is the usual one; only the label changed. `LogisticRegression(max_iter=1000)` inside it.

</details>

<details>
<summary>✅ Solution</summary>

```python
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
accuracy = round(accuracy_score(test['move'],
                                three.predict(test[['vol_20d', 'ret_20d']])), 4)

print(accuracy)
```

0.5539, against 0.3506 for the baseline in G2. Nothing in the call changed: `LogisticRegression` saw three values in the label and fitted three classes.

</details>

---

### G4 · What the fitted model grew  ★★☆☆☆

Continuing from G3. Print `classes_`, the shape of `coef_` and the shape of `intercept_`. Reach inside the pipeline with `named_steps`.

In [ ]:
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])

classes = ...
coef_shape = ...
intercept_shape = ...

print(classes)
print(coef_shape)
print(intercept_shape)

<details>
<summary>💡 Hint</summary>

`three.named_steps['logit']`, as in Session 6.

</details>

<details>
<summary>✅ Solution</summary>

```python
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])

logit = three.named_steps['logit']
classes = logit.classes_
coef_shape = logit.coef_.shape
intercept_shape = logit.intercept_.shape

print(classes)
print(coef_shape)
print(intercept_shape)
```

`['falls', 'rises', 'stays']`, `coef_` of shape (3, 2) and `intercept_` of shape (3,). The classes come back in alphabetical order, not the order the cuts were written in, and `coef_` has one row per class rather than one row in total.

</details>

---

### G5 · Three scores for one day  ★★★☆☆  · revisits S5

Continuing from G3. Take the first test day and print the three raw scores from `decision_function`, rounded to three decimals, beside `classes_` so you can tell which is which.

In [ ]:
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
day = test[['vol_20d', 'ret_20d']].head(1)

scores = ...

print(three.named_steps['logit'].classes_)
print(scores)

<details>
<summary>💡 Hint</summary>

`three.decision_function(day)[0]`. The `[0]` takes the single row out of the result.

</details>

<details>
<summary>✅ Solution</summary>

```python
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
day = test[['vol_20d', 'ret_20d']].head(1)

scores = three.decision_function(day)[0].round(3)

print(three.named_steps['logit'].classes_)
print(scores)
```

`[0.06, -0.324, 0.264]`. These are not probabilities: one is negative, and they do not add to one. G6 turns them into probabilities.

</details>

---

### G6 · The softmax, by hand  ★★★★☆  · revisits S3

Continuing from G5. Turn the three scores into three probabilities with the exponential and a division, then check your answer against `predict_proba`.

In [ ]:
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
day = test[['vol_20d', 'ret_20d']].head(1)
scores = three.decision_function(day)[0]

raised = ...
probabilities = ...

print(probabilities)
print(three.predict_proba(day).round(3))

<details>
<summary>💡 Hint 1</summary>

`np.exp(scores)` makes every number positive.

</details>

<details>
<summary>💡 Hint 2</summary>

Divide by their total: `raised / raised.sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
day = test[['vol_20d', 'ret_20d']].head(1)
scores = three.decision_function(day)[0]

raised = np.exp(scores)
probabilities = (raised / raised.sum()).round(3)

print(probabilities)
print(three.predict_proba(day).round(3))
```

Both print `[0.344, 0.234, 0.422]`. The exponential turns [0.06, -0.324, 0.264] into [1.061, 0.724, 1.302], which total 3.087, and dividing by that total makes them add to one. With two classes the same two steps are the sigmoid.

</details>

---

### G7 · The confusion matrix, in a readable order  ★★★★☆

Continuing from G3. Print the confusion matrix with the rows and columns in the order `falls`, `stays`, `rises` rather than alphabetically, and count how many days were wrong by two steps.

In [ ]:
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
predicted = three.predict(test[['vol_20d', 'ret_20d']])
order = ['falls', 'stays', 'rises']

cm = ...
two_steps = ...

print(cm)
print(two_steps)

<details>
<summary>💡 Hint 1</summary>

`confusion_matrix(test['move'], predicted, labels=order)`.

</details>

<details>
<summary>💡 Hint 2</summary>

A two-step error is a fall called a rise or a rise called a fall: `cm[0, 2] + cm[2, 0]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
predicted = three.predict(test[['vol_20d', 'ret_20d']])
order = ['falls', 'stays', 'rises']

cm = confusion_matrix(test['move'], predicted, labels=order)
two_steps = cm[0, 2] + cm[2, 0]

print(cm)
print(two_steps)
```

17 of the 482 test days are wrong by two steps. Without `labels=` the matrix comes back alphabetically, with `rises` in the middle, and the diagonal then means something quite different from what you expect.

</details>

---

### G8 · One score per class  ★★☆☆☆

Print `classification_report` for the three-class model, with the same order and three decimals, then print the macro and weighted F1 separately.

In [ ]:
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
predicted = three.predict(test[['vol_20d', 'ret_20d']])
order = ['falls', 'stays', 'rises']

print(classification_report(test['move'], predicted, labels=order, digits=3))

macro = ...
weighted = ...

print(macro)
print(weighted)

<details>
<summary>💡 Hint</summary>

`f1_score(test['move'], predicted, average='macro')` and the same with `average='weighted'`.

</details>

<details>
<summary>✅ Solution</summary>

```python
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
predicted = three.predict(test[['vol_20d', 'ret_20d']])
order = ['falls', 'stays', 'rises']

print(classification_report(test['move'], predicted, labels=order, digits=3))

macro = round(f1_score(test['move'], predicted, average='macro'), 3)
weighted = round(f1_score(test['move'], predicted, average='weighted'), 3)

print(macro)
print(weighted)
```

Macro 0.551, weighted 0.528. Recall is 0.734 for `falls` and 0.857 for `rises`, but only 0.293 for `stays`: the model finds the two ends and struggles in the middle. One accuracy would have hidden that completely.

</details>

---

### G9 · Draw the recall per class  ★★☆☆☆  · revisits S3

Draw the recall of each of the three classes as a bar chart, in the order `falls`, `stays`, `rises`, with a dashed line at the accuracy of the whole model.

In [ ]:
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
predicted = three.predict(test[['vol_20d', 'ret_20d']])
order = ['falls', 'stays', 'rises']

from sklearn.metrics import recall_score
recalls = recall_score(test['move'], predicted, average=None, labels=order)

fig, ax = plt.subplots(figsize=(5, 3))
...
...
ax.set_ylabel('recall')
ax.set_ylim(0, 1)
plt.show()

<details>
<summary>💡 Hint</summary>

`ax.bar(order, recalls)` and `ax.axhline(accuracy_score(test['move'], predicted), linestyle='--', color='grey')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
predicted = three.predict(test[['vol_20d', 'ret_20d']])
order = ['falls', 'stays', 'rises']

from sklearn.metrics import recall_score
recalls = recall_score(test['move'], predicted, average=None, labels=order)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(order, recalls)
ax.axhline(accuracy_score(test['move'], predicted), linestyle='--', color='grey')
ax.set_ylabel('recall')
ax.set_ylim(0, 1)
ax.set_title('Recall by class', loc='left')
plt.show()
```

Two bars above the line and one well below it: 0.73 for `falls`, 0.86 for `rises` and 0.29 for `stays`. The single accuracy is the dashed line, and it describes none of the three.

</details>

---

---

## H · k-nearest neighbours

A classifier with no coefficients. H3 and H4 run together.

### H1 · The four lines, with a different model  ★★☆☆☆

Fit `KNeighborsClassifier` with 15 neighbours inside a scaled pipeline, on `vol_20d` and `ret_20d` predicting `rising`, and print the test AUC.

In [ ]:
neighbours = ...
...
p_knn = ...
auc = ...

print(auc)

<details>
<summary>💡 Hint</summary>

`Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=15))])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
neighbours = Pipeline([('scale', StandardScaler()),
                       ('knn', KNeighborsClassifier(n_neighbors=15))])
neighbours.fit(train[['vol_20d', 'ret_20d']], train['rising'])
p_knn = neighbours.predict_proba(test[['vol_20d', 'ret_20d']])[:, 1]
auc = round(roc_auc_score(test['rising'], p_knn), 3)

print(auc)
```

0.634. The four lines are the ones every model in this course takes, with a different class on the first. Nothing about the scoring changes either.

</details>

---

### H2 · Why the scaler is not optional  ★★★☆☆  · revisits S6

On the credit table, fit k-nearest neighbours with 25 neighbours twice, once with a scaler in front and once without, and print both test AUCs. Then print the standard deviation of `limit` and of `late_now`.

In [ ]:
bare = ...
scaled = ...
aucs = ...

print(aucs)
print(round(c_train['limit'].std(), 0), round(c_train['late_now'].std(), 2))

<details>
<summary>💡 Hint 1</summary>

`bare = KNeighborsClassifier(n_neighbors=25)` with no pipeline at all.

</details>

<details>
<summary>💡 Hint 2</summary>

`scaled = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=25))])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
bare = KNeighborsClassifier(n_neighbors=25)
scaled = Pipeline([('scale', StandardScaler()),
                   ('knn', KNeighborsClassifier(n_neighbors=25))])
aucs = []
for model in [bare, scaled]:
    model.fit(c_train[c_cols], c_train['default'])
    aucs.append(round(float(roc_auc_score(c_test['default'],
                model.predict_proba(c_test[c_cols])[:, 1])), 3))

print(aucs)
print(round(c_train['limit'].std(), 0), round(c_train['late_now'].std(), 2))
```

0.627 without the scaler and 0.749 with it. `limit` varies by about 128,965 and `late_now` by 0.76, so an unscaled distance between two borrowers is the difference in their credit limits and nothing else. Ridge needed the scaler because of its penalty; this needs it because of the distance.

</details>

---

### H3 · k as the dial  ★★★☆☆  · revisits S2

Loop over k values 1, 15, 51, 151 and 301 and print each one with the TRAINING AUC and the TEST AUC. Keep the loop: H4 continues the idea.

In [ ]:
for k in [1, 15, 51, 151, 301]:
    model = Pipeline([('scale', StandardScaler()),
                      ('knn', KNeighborsClassifier(n_neighbors=k))])
    model.fit(train[['vol_20d', 'ret_20d']], train['rising'])
    train_auc = ...
    test_auc = ...
    print(k, train_auc, test_auc)

<details>
<summary>💡 Hint</summary>

Build the pipeline inside the loop with `KNeighborsClassifier(n_neighbors=k)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for k in [1, 15, 51, 151, 301]:
    model = Pipeline([('scale', StandardScaler()),
                      ('knn', KNeighborsClassifier(n_neighbors=k))])
    model.fit(train[['vol_20d', 'ret_20d']], train['rising'])
    train_auc = round(roc_auc_score(train['rising'],
                      model.predict_proba(train[['vol_20d', 'ret_20d']])[:, 1]), 3)
    test_auc = round(roc_auc_score(test['rising'],
                     model.predict_proba(test[['vol_20d', 'ret_20d']])[:, 1]), 3)
    print(k, train_auc, test_auc)
```

At k of 1 the training AUC is 1.000 and the test AUC is 0.572: every training day is its own nearest neighbour, so the model reproduces the rows it learned from and knows nothing else. By k of 301 they are 0.688 and 0.753. A large k is a simple model, in the same way that a small `C` was.

</details>

---

### H4 · Choosing k on the folds  ★★★★☆  · revisits S6

Continuing from H3. Search k over 1, 5, 15, 51, 101, 151, 201 and 301 with `GridSearchCV`, the time folds and `scoring='roc_auc'`, and print the winner with its mean score.

In [ ]:
neighbours = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier())])

grid = ...
search = ...
...
best = ...

print(best)

<details>
<summary>💡 Hint 1</summary>

The grid key is the step name, two underscores, the argument: `{'knn__n_neighbors': [1, 5, 15, 51, 101, 151, 201, 301]}`.

</details>

<details>
<summary>💡 Hint 2</summary>

`GridSearchCV(neighbours, grid, cv=folds, scoring='roc_auc')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
neighbours = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier())])

grid = {'knn__n_neighbors': [1, 5, 15, 51, 101, 151, 201, 301]}
search = GridSearchCV(neighbours, grid, cv=folds, scoring='roc_auc')
search.fit(train[['vol_20d', 'ret_20d']], train['rising'])
best = (search.best_params_, round(float(search.best_score_), 3))

print(best)
```

k of 301, with a mean AUC of 0.717 over the five folds. The search is the one used for `alpha` and for `C`, with a different setting named in the grid.

</details>

---

### H5 · The k the folds cannot take  ★☆☆☆☆

Print the number of fitting rows in each of the five time folds. Then say why a grid containing k of 401 would return `nan` for that value.

In [ ]:
for fit_rows, score_rows in folds.split(train):
    ...

<details>
<summary>💡 Hint</summary>

`print(len(fit_rows), len(score_rows))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for fit_rows, score_rows in folds.split(train):
    print(len(fit_rows), len(score_rows))
```

The first fold fits on only 319 days, so a k of 401 asks for more neighbours than there are rows to find them in. `GridSearchCV` turns the failure into `nan` and a warning rather than stopping, so the value silently drops out of the search. The limit is the smallest fold, not the size of the training rows.

</details>

---

### H6 · k-NN against logistic regression  ★★★☆☆  · revisits S5

Cross-validate a scaled logistic regression on the same two columns and the same folds, and print its mean AUC. Compare it with the 0.717 the best k reached in H4.

In [ ]:
logistic = ...
scores = ...
mean_auc = ...

print(scores)
print(mean_auc)

<details>
<summary>💡 Hint</summary>

`Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
logistic = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
scores = cross_val_score(logistic, train[['vol_20d', 'ret_20d']],
                         train['rising'], cv=folds, scoring='roc_auc').round(3)
mean_auc = round(float(scores.mean()), 3)

print(scores)
print(mean_auc)
```

0.745 against 0.717, so the folds choose logistic regression. On the test days the gap is wider: 0.847 against 0.753. A flexible model is not automatically a better one.

</details>

---

### H7 · The probabilities a vote can give  ★★★☆☆  · revisits S2

Fit k-nearest neighbours with 5 neighbours on the two columns and print the distinct probabilities it produces on the test days, in order. Use `sorted` and `set`.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
model.fit(train[['vol_20d', 'ret_20d']], train['rising'])
p_knn = model.predict_proba(test[['vol_20d', 'ret_20d']])[:, 1]

distinct = ...

print(distinct)

<details>
<summary>💡 Hint</summary>

`p_knn.round(3).tolist()` turns the array into a plain list, then `sorted(set(...))` drops the duplicates and puts the rest in order.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
model.fit(train[['vol_20d', 'ret_20d']], train['rising'])
p_knn = model.predict_proba(test[['vol_20d', 'ret_20d']])[:, 1]

distinct = sorted(set(p_knn.round(3).tolist()))

print(distinct)
```

Six values: [np.float64(0.0), np.float64(0.2), np.float64(0.4), np.float64(0.6), np.float64(0.8), np.float64(1.0)]. A vote among 5 neighbours can only ever give a multiple of one fifth, so the probabilities are coarse. A larger k gives finer ones, which is a second reason beyond smoothness to prefer it.

</details>

---

### H8 · Draw the two AUCs against k  ★★☆☆☆  · revisits S6

Draw the training AUC and the test AUC against k as two lines, with k on a logarithmic axis and a legend naming them. The numbers are the ones from H3, over a few more values of k.

In [ ]:
ks = [1, 3, 9, 25, 51, 101, 201, 301]
train_aucs = []
test_aucs = []
for k in ks:
    model = Pipeline([('scale', StandardScaler()),
                      ('knn', KNeighborsClassifier(n_neighbors=k))])
    model.fit(train[['vol_20d', 'ret_20d']], train['rising'])
    train_aucs.append(roc_auc_score(train['rising'],
                      model.predict_proba(train[['vol_20d', 'ret_20d']])[:, 1]))
    test_aucs.append(roc_auc_score(test['rising'],
                     model.predict_proba(test[['vol_20d', 'ret_20d']])[:, 1]))

fig, ax = plt.subplots(figsize=(7, 3))
...
...
...
...
ax.set_xlabel('k')
ax.set_ylabel('AUC')
plt.show()

<details>
<summary>💡 Hint 1</summary>

Two `ax.plot(ks, ..., label=...)` calls, then `ax.set_xscale('log')` and `ax.legend()`.

</details>

<details>
<summary>💡 Hint 2</summary>

A logarithmic axis is right here because the k values are spaced by factors rather than by steps, exactly as the alphas were in Session 6.

</details>

<details>
<summary>✅ Solution</summary>

```python
ks = [1, 3, 9, 25, 51, 101, 201, 301]
train_aucs = []
test_aucs = []
for k in ks:
    model = Pipeline([('scale', StandardScaler()),
                      ('knn', KNeighborsClassifier(n_neighbors=k))])
    model.fit(train[['vol_20d', 'ret_20d']], train['rising'])
    train_aucs.append(roc_auc_score(train['rising'],
                      model.predict_proba(train[['vol_20d', 'ret_20d']])[:, 1]))
    test_aucs.append(roc_auc_score(test['rising'],
                     model.predict_proba(test[['vol_20d', 'ret_20d']])[:, 1]))

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(ks, train_aucs, marker='o', label='training rows')
ax.plot(ks, test_aucs, marker='o', label='test rows')
ax.set_xscale('log')
ax.legend()
ax.set_xlabel('k')
ax.set_ylabel('AUC')
ax.set_title('A small k fits the training rows and nothing else', loc='left')
plt.show()
```

The two lines start far apart, at 1.00 and 0.57, and close as k grows. This is the same picture the validation curve drew for alpha in Session 6 and for C in Session 8, with a different setting on the horizontal axis.

</details>

---

---

## I · When it goes wrong

Five mistakes that are easy to make and quiet when they happen.

### I1 · AUC on the wrong argument  ★☆☆☆☆

This cell passes the 0/1 predictions to `roc_auc_score` instead of the probabilities. Run it, read the number, then fix it in the cell below.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
predicted = model.predict(c_test[c_cols])

print(round(roc_auc_score(c_test['default'], predicted), 4))

<details>
<summary>💡 Hint</summary>

Nothing raises here. That is the problem: the number is wrong but the code runs.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(c_train[c_cols], c_train['default'])
p = model.predict_proba(c_test[c_cols])[:, 1]

print(round(roc_auc_score(c_test['default'], p), 4))
```

With the predictions it reports about 0.6347; with the probabilities, 0.7476. Passing 0s and 1s throws away the ranking and leaves the AUC reading a single threshold. No error is raised, so this one is only caught by knowing what the function wants.

</details>

---

### I2 · A pipeline that scales nothing  ★★☆☆☆  · revisits S6

Write the fix: the cell below fits k-nearest neighbours directly on the credit columns, with no scaler. Rewrite it as a pipeline that scales first, and print both AUCs.

In [ ]:
wrong = KNeighborsClassifier(n_neighbors=25)
wrong.fit(c_train[c_cols], c_train['default'])
print(round(roc_auc_score(c_test['default'], wrong.predict_proba(c_test[c_cols])[:, 1]), 3))

right = ...
...
auc_right = ...

print(auc_right)

<details>
<summary>💡 Hint</summary>

`right = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=25))])`, then `right.fit(...)` on the next line.

</details>

<details>
<summary>✅ Solution</summary>

```python
wrong = KNeighborsClassifier(n_neighbors=25)
wrong.fit(c_train[c_cols], c_train['default'])
print(round(roc_auc_score(c_test['default'], wrong.predict_proba(c_test[c_cols])[:, 1]), 3))

right = Pipeline([('scale', StandardScaler()),
                  ('knn', KNeighborsClassifier(n_neighbors=25))])
right.fit(c_train[c_cols], c_train['default'])
auc_right = round(roc_auc_score(c_test['default'],
                                right.predict_proba(c_test[c_cols])[:, 1]), 3)

print(auc_right)
```

0.627 becomes 0.749. Nothing warned you. The pipeline exists so the scaler cannot be forgotten and cannot be fitted on the test rows by accident.

</details>

---

### I3 · A shuffled split on a time series  ★★☆☆☆

Run this cell, which splits the index table at random, and print the first and last date on each side. Then say in a comment why the score that follows would be meaningless.

In [ ]:
early, late = train_test_split(table, test_size=0.3, random_state=0)

print(early.index.min().date(), early.index.max().date())
print(late.index.min().date(), late.index.max().date())

# why is this wrong here?
...

<details>
<summary>💡 Hint</summary>

Look at the date ranges. Do they overlap?

</details>

<details>
<summary>✅ Solution</summary>

```python
early, late = train_test_split(table, test_size=0.3, random_state=0)

print(early.index.min().date(), early.index.max().date())
print(late.index.min().date(), late.index.max().date())

# Both halves run from 2015 to 2024, so the model would be fitted on days
# that come after the days it is scored on. Neighbouring days also share
# most of a 20-day window, so a shuffled split puts nearly the same row on
# both sides.
```

Both halves cover the whole period. `train_test_split` is right for the credit table, where rows are separate borrowers, and wrong here, where rows are days in order and overlapping windows make neighbouring rows nearly copies of each other.

</details>

---

### I4 · A confusion matrix in the wrong order  ★★☆☆☆  · revisits S3

Print the three-class confusion matrix twice, once without `labels=` and once with `labels=['falls', 'stays', 'rises']`, and print the model's `classes_` between them.

In [ ]:
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
predicted = three.predict(test[['vol_20d', 'ret_20d']])

print(confusion_matrix(test['move'], predicted))
print(three.named_steps['logit'].classes_)
print(...)

<details>
<summary>💡 Hint</summary>

`confusion_matrix(test['move'], predicted, labels=['falls', 'stays', 'rises'])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])
predicted = three.predict(test[['vol_20d', 'ret_20d']])

print(confusion_matrix(test['move'], predicted))
print(three.named_steps['logit'].classes_)
print(confusion_matrix(test['move'], predicted, labels=['falls', 'stays', 'rises']))
```

The first matrix is in alphabetical order, so its middle row and column are `rises`, not `stays`. Both matrices are correct; only one of them means what you would assume at a glance. Read `classes_` before reading any matrix.

</details>

---

### I5 · Two columns or three  ★★☆☆☆  · revisits S8

Print the shape of `predict_proba` for the two-class model on `rising` and for the three-class model on `move`, and the sum of the first row of each.

In [ ]:
two = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
two.fit(train[['vol_20d', 'ret_20d']], train['rising'])
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])

for model in [two, three]:
    probabilities = model.predict_proba(test[['vol_20d', 'ret_20d']])
    ...

<details>
<summary>💡 Hint</summary>

`print(probabilities.shape, round(float(probabilities[0].sum()), 6))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
two = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
two.fit(train[['vol_20d', 'ret_20d']], train['rising'])
three = Pipeline([('scale', StandardScaler()),
                  ('logit', LogisticRegression(max_iter=1000))])
three.fit(train[['vol_20d', 'ret_20d']], train['move'])

for model in [two, three]:
    probabilities = model.predict_proba(test[['vol_20d', 'ret_20d']])
    print(probabilities.shape, round(float(probabilities[0].sum()), 6))
```

(482, 2) and (482, 3), both summing to 1 on every row. The habit of writing `[:, 1]` comes from the two-class case, where the second column is the probability of a 1. With three classes there is no single column to take, and `[:, 1]` silently gives you the second class in alphabetical order.

</details>

---

---

## J · The rare label across the desk

J1 writes a function; J2 to J5 all use it. The label is the rare one: volatility over the next 20 days more than 1.5 times this month's.

### J1 · A function that builds the table  ★★★★☆  · revisits S2

Write `jump_table(ticker)` returning a table with `vol_20d`, `vol_next` and the label `jump`, for one instrument, with the incomplete rows dropped. `rets` from the setup cell holds the returns in percent.

In [ ]:
def jump_table(ticker):
    """..."""
    ...


print(jump_table('AAPL'))

<details>
<summary>💡 Hint 1</summary>

The target looks forward: `r.rolling(20).std().shift(-20)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`(frame['vol_next'] > 1.5 * frame['vol_20d']).astype(int)`, made after the dropna so the two columns line up.

</details>

<details>
<summary>✅ Solution</summary>

```python
def jump_table(ticker):
    """vol_20d, vol_next and a rare 0/1 jump label for one instrument."""
    r = rets[ticker]
    frame = pd.DataFrame({'vol_20d': r.rolling(20).std()})
    frame['vol_next'] = r.rolling(20).std().shift(-20)
    frame = frame.dropna()
    frame['jump'] = (frame['vol_next'] > 1.5 * frame['vol_20d']).astype(int)
    return frame


print(jump_table('AAPL'))
```

One function, eleven instruments. Writing the table-building code once and calling it in a loop is what makes the next four exercises short, and it is the same move as `build_table` in Session 8.

</details>

---

### J2 · The share of jumps per instrument  ★★☆☆☆  · revisits S2

Using `jump_table` from J1, build a dictionary from ticker to the share of TEST days that were jumps, and print it.

In [ ]:
shares = {}
for ticker in tickers:
    ...

print(shares)

<details>
<summary>💡 Hint</summary>

`shares[ticker] = round(float(late['jump'].mean()), 3)`. `float()` keeps the printed dictionary readable.

</details>

<details>
<summary>✅ Solution</summary>

```python
shares = {}
for ticker in tickers:
    late = jump_table(ticker).loc['2023-01-01':]
    shares[ticker] = round(float(late['jump'].mean()), 3)

print(shares)
```

From 0.035 to 0.247. The same definition of a jump gives a class of quite different rarity on each instrument, and the baseline to beat moves with it.

</details>

---

### J3 · The AUC per instrument  ★★★☆☆  · revisits S5

Fit the one-column classifier on each instrument's training days and collect the test AUC in a dictionary. Print it sorted, largest first.

In [ ]:
aucs = {}
for ticker in tickers:
    ...

for ticker in sorted(aucs, key=aucs.get, reverse=True):
    print(ticker, aucs[ticker])

<details>
<summary>💡 Hint</summary>

`aucs[ticker] = round(float(roc_auc_score(late['jump'], model.predict_proba(late[['vol_20d']])[:, 1])), 3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
aucs = {}
for ticker in tickers:
    frame = jump_table(ticker)
    early = frame.loc[:'2022-12-31']
    late = frame.loc['2023-01-01':]
    model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
    model.fit(early[['vol_20d']], early['jump'])
    aucs[ticker] = round(float(roc_auc_score(late['jump'],
                        model.predict_proba(late[['vol_20d']])[:, 1])), 3)

for ticker in sorted(aucs, key=aucs.get, reverse=True):
    print(ticker, aucs[ticker])
```

MSFT ranks best at 0.972 and JPM worst at 0.610. Eight of the eleven are above 0.74, so this month's volatility really does say something about whether next month jumps.

</details>

---

### J4 · How often the model says nothing  ★★★☆☆  · revisits S2

Count how many of the eleven instruments get NO predicted jump at all at a threshold of one half, and print the count with the list of tickers.

In [ ]:
silent = []
for ticker in tickers:
    ...

print(len(silent))
print(silent)

<details>
<summary>💡 Hint</summary>

`if predicted.sum() == 0:` is the condition for a model that never predicts a jump.

</details>

<details>
<summary>✅ Solution</summary>

```python
silent = []
for ticker in tickers:
    frame = jump_table(ticker)
    early = frame.loc[:'2022-12-31']
    late = frame.loc['2023-01-01':]
    model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
    model.fit(early[['vol_20d']], early['jump'])
    predicted = model.predict(late[['vol_20d']])
    if predicted.sum() == 0:
        silent.append(ticker)

print(len(silent))
print(silent)
```

9 of the 11. On those instruments the model scores exactly the majority rule, catches nothing, and looks respectable while doing it. This is the single most useful thing to check on a rare class, and it takes one line.

</details>

---

### J5 · The report line  ★★★★★  · revisits S5

Write a loop that prints one line per instrument with the ticker, the share of jumps and the AUC, lined up with field widths and sorted by AUC. Reuse `shares` from J2 and `aucs` from J3, or rebuild them.

In [ ]:
shares = {}
aucs = {}
for ticker in tickers:
    ...

print(f"{'ticker':<8}{'share':>8}{'AUC':>8}")
for ticker in sorted(aucs, key=aucs.get, reverse=True):
    print(f"{ticker:<8}{shares[ticker]:>8.3f}{aucs[ticker]:>8.3f}")

<details>
<summary>💡 Hint</summary>

Loop over `sorted(aucs, key=aucs.get, reverse=True)` and print `f"{ticker:<8}{shares[ticker]:>8.3f}{aucs[ticker]:>8.3f}"`.

</details>

<details>
<summary>✅ Solution</summary>

```python
shares = {}
aucs = {}
for ticker in tickers:
    frame = jump_table(ticker)
    early = frame.loc[:'2022-12-31']
    late = frame.loc['2023-01-01':]
    model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
    model.fit(early[['vol_20d']], early['jump'])
    shares[ticker] = late['jump'].mean()
    aucs[ticker] = roc_auc_score(late['jump'], model.predict_proba(late[['vol_20d']])[:, 1])

print(f"{'ticker':<8}{'share':>8}{'AUC':>8}")
for ticker in sorted(aucs, key=aucs.get, reverse=True):
    print(f"{ticker:<8}{shares[ticker]:>8.3f}{aucs[ticker]:>8.3f}")
```

A dictionary, a `sorted` with `key=`, and three field widths. That is the whole of a desk report, and every piece of it came from Sessions 2 and 5.

</details>

---

---

## K · Five small cases

Each of these stands completely on its own and needs no model from earlier in the notebook. They are here to keep loops, `if`, dictionaries, f-strings and functions in working order, in this session's setting.

### K1 · Turning probabilities into advice  ★★☆☆☆  · revisits S2

A model has given seven days these probabilities of a jump. Loop over them and print one line each: `low` below 0.2, `act` at 0.5 or above, and `watch` in between. Print the probability as a percentage with no decimals, beside the word.

In [ ]:
probabilities = [0.04, 0.19, 0.33, 0.51, 0.08, 0.62, 0.27]

for value in probabilities:
    ...

<details>
<summary>💡 Hint 1</summary>

An `if`, an `elif` and an `else`, in that order, inside the loop.

</details>

<details>
<summary>💡 Hint 2</summary>

`print(f"{value:.0%}", word)` once the word is decided.

</details>

<details>
<summary>✅ Solution</summary>

```python
probabilities = [0.04, 0.19, 0.33, 0.51, 0.08, 0.62, 0.27]

for value in probabilities:
    if value < 0.2:
        word = 'low'
    elif value >= 0.5:
        word = 'act'
    else:
        word = 'watch'
    print(f"{value:.0%}", word)
```

The seven come out as low, low, watch, act, low, act, watch. Two thresholds and three words: a classifier's output becomes a decision only once someone writes down rules like these, and they are ordinary `if` statements.

</details>

---

### K2 · A confusion matrix with no library  ★★★☆☆  · revisits S2

Two lists hold what a model said and what happened, for eight days. Count the four combinations into a dictionary with a single loop over `zip`, and print it.

In [ ]:
said = [1, 0, 1, 1, 0, 0, 1, 0]
was  = [1, 0, 0, 1, 1, 0, 1, 1]

counts = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}
for s, w in zip(said, was):
    ...

print(counts)

<details>
<summary>💡 Hint 1</summary>

Four cases: `if s == 1 and w == 1:` is a true positive.

</details>

<details>
<summary>💡 Hint 2</summary>

`counts['tp'] += 1` adds one to that entry.

</details>

<details>
<summary>✅ Solution</summary>

```python
said = [1, 0, 1, 1, 0, 0, 1, 0]
was  = [1, 0, 0, 1, 1, 0, 1, 1]

counts = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}
for s, w in zip(said, was):
    if s == 1 and w == 1:
        counts['tp'] += 1
    elif s == 1 and w == 0:
        counts['fp'] += 1
    elif s == 0 and w == 1:
        counts['fn'] += 1
    else:
        counts['tn'] += 1

print(counts)
```

`{'tp': 3, 'fp': 1, 'fn': 2, 'tn': 2}`. `confusion_matrix` does exactly this and nothing more. `zip` walks the two lists in step, which is the right tool whenever two sequences line up row by row.

</details>

---

### K3 · A sentence about one instrument  ★★☆☆☆  · revisits S2

Write `verdict(ticker, auc, share)` returning a single sentence, and call it twice. The sentence should name the instrument, give the AUC to two decimals and the share as a percentage, and end with `worth using` when the AUC is at least 0.75 and `not yet` otherwise.

In [ ]:
def verdict(ticker, auc, share):
    """..."""
    ...


print(verdict('AAPL', 0.889, 0.087))
print(verdict('JPM', 0.610, 0.160))

<details>
<summary>💡 Hint 1</summary>

Decide the ending with an `if` and store it in a variable.

</details>

<details>
<summary>💡 Hint 2</summary>

Return an f-string: `f"{ticker}: AUC {auc:.2f} on {share:.0%} jumps, {ending}"`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def verdict(ticker, auc, share):
    """One sentence about one instrument's jump model."""
    if auc >= 0.75:
        ending = 'worth using'
    else:
        ending = 'not yet'
    return f"{ticker}: AUC {auc:.2f} on {share:.0%} jumps, {ending}"


print(verdict('AAPL', 0.889, 0.087))
print(verdict('JPM', 0.610, 0.160))
```

`AAPL: AUC 0.89 on 9% jumps, worth using` and `JPM: AUC 0.61 on 16% jumps, not yet`. A function that returns a string rather than printing it can be used in a loop, put in a list, or written to a file. Printing inside the function would close all three doors.

</details>

---

### K4 · The longest run of warnings  ★★★★☆  · revisits S2

Fit the one-column jump model on the index table, call a jump whenever the probability reaches 0.3, and find the longest run of consecutive called days. Use a counter and a `for` loop.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(train[['vol_20d']], train['jump'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
called = (p >= 0.3).astype(int)

run = 0
longest = 0
for value in called:
    ...

print(called.sum())
print(longest)

<details>
<summary>💡 Hint 1</summary>

Inside the loop: if the value is 1, add one to `run`; otherwise reset `run` to 0.

</details>

<details>
<summary>💡 Hint 2</summary>

After updating `run`, keep the best: `longest = max(longest, run)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(train[['vol_20d']], train['jump'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
called = (p >= 0.3).astype(int)

run = 0
longest = 0
for value in called:
    if value == 1:
        run += 1
    else:
        run = 0
    longest = max(longest, run)

print(called.sum())
print(longest)
```

18 days called in total, and the longest unbroken run is 13. The warnings arrive in clusters rather than spread out, which is what you would expect from a column that moves slowly. A counter that resets is the standard shape for the longest run of anything.

</details>

---

### K5 · Accuracy, year by year  ★★★☆☆  · revisits S3

Fit the one-column jump model, predict the test days, and compute the accuracy separately for 2023 and 2024 with `groupby`: make a Series of `True` and `False` indexed by date, and group it by year.

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(train[['vol_20d']], train['jump'])
predicted = model.predict(test[['vol_20d']])

right = ...
by_year = ...

print(by_year)

<details>
<summary>💡 Hint 1</summary>

`right = pd.Series(predicted == test['jump'].values, index=test.index)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`right.groupby(right.index.year).mean()`: the mean of a column of `True` and `False` is the share of `True`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
model.fit(train[['vol_20d']], train['jump'])
predicted = model.predict(test[['vol_20d']])

right = pd.Series(predicted == test['jump'].values, index=test.index)
by_year = right.groupby(right.index.year).mean()

print(by_year)
```

0.980 in 2023 and 0.866 in 2024. The model predicts no jump on any day, so these two numbers are just the share of days with no jump in each year: 2024 had more jumps, and the accuracy fell without the model changing at all. One number over two years would have hidden that.

</details>

---

## 🏁 Done

You built rare labels and the baselines they have to beat, split a cross-section at random and saw why a time series cannot be, read the four counts by hand and from the library, priced the two kinds of mistake and found the threshold that follows from those prices, weighted a rare class and watched the probabilities stop meaning anything, put them right again, fitted three classes and turned three scores into three probabilities, and lost a fair fight between k-nearest neighbours and logistic regression.

The case takes the threshold and the rare label back to the risk report, where Apple's jump model scores 91 percent accuracy and predicts nothing at all.